# UNSUPERVISED PART : CVRIE

In [17]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

print('Numpy:', np.__version__)
print('Pandas:', pd.__version__)

Numpy: 2.4.2
Pandas: 3.0.1


In [18]:
csv_path = "../Student_Dataset.csv"
dataframe = pd.read_csv(csv_path, encoding='utf-8-sig')
print(dataframe.head())
print(dataframe.shape)
print(dataframe.info())


                                                   1  \
2                                           0x000000   
3                                           0x000000   
4  A short sprint brought on a twinge in the midd...   
5                                           0x000000   
6  A sudden twist while reaching for a bag made o...   

  On uneven paths the heel feels wobbly side‑to‑side. Slanted pavements bring a quick aching shift under the ankle when I step.  
2  The feverish feeling eased, but I’m now breath...                                                                             
3  While working in the garden, I touched a metal...                                                                             
4                                                NaN                                                                             
5  I accidentally left my hand on a hot mug for a...                                                                             
6            

# EDA 


on peut constater que ce dataset contient 1010 lignes et 2 colonnes mais lorsque'on regarde le cvs de plus pret on peut voir des hexadecimal parfois :

C'est la conséquence des virgules dans les témoignages qui sont mal parser donc il faut corriger ca.

In [19]:
csv_path = "../Student_Dataset.csv"

tempdf = pd.read_csv(csv_path, encoding='utf-8-sig', header=None,
    names=['id', 'col2', 'col3'])

tempdf['text'] = np.where(tempdf['col2'] == '0x000000', tempdf['col3'], tempdf['col2'])
dataframe = tempdf[['id', 'text']]



print(dataframe.head())
print(dataframe.shape)
print(dataframe.info())
print('nombres elements louche dans le df: ', dataframe['text'].str.contains('0x000000').sum())


   id                                               text
0   1  On uneven paths the heel feels wobbly side‑to‑...
1   2  The feverish feeling eased, but I’m now breath...
2   3  While working in the garden, I touched a metal...
3   4  A short sprint brought on a twinge in the midd...
4   5  I accidentally left my hand on a hot mug for a...
(1011, 2)
<class 'pandas.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      1011 non-null   int64
 1   text    1011 non-null   str  
dtypes: int64(1), str(1)
memory usage: 15.9 KB
None
nombres elements louche dans le df:  0


# PreProcessing

Maitenant que nous avons enlever la 3 eme colonne aléatoire a cause du mauvais parsing du csv on peut commencer a s'occuper des stop words : Les motes qui n'ont pas d'utilité pour lié les différents témoiganges en eux :


## Les Stops Words

Comme les témopignages sont en anglais on peut utilisé la liste de stopwords contenue  avec sklearn

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
import re

vectorizer = TfidfVectorizer(stop_words='english')
stop_words = vectorizer.get_stop_words()

def remove_stop_words(text):
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

import re

def remove_numbers(text):
    return re.sub(r'\d+\.?\d*\S*', '', text)



dataframe['clean_text'] = dataframe['text'].apply(remove_stop_words)
dataframe['clean_text'] = dataframe['clean_text'].apply(remove_numbers)

print(dataframe[['text', 'clean_text']].head(3))


                                                text  \
0  On uneven paths the heel feels wobbly side‑to‑...   
1  The feverish feeling eased, but I’m now breath...   
2  While working in the garden, I touched a metal...   

                                          clean_text  
0  uneven paths heel feels wobbly side‑to‑side. s...  
1  feverish feeling eased, i’m breathing faster d...  
2  working garden, touched metal grill stove. lef...  


ici on crée une nouvelle colonne clean_text car on veut grader la colonne original pour éventuels debug .

aussi on oublie pas d'enlever les nombres (et le petit float caché)

## Lemmatisation 

On constate que dans les note du kickoff on avait parler de TF idf qui est une mthode de vectoring que l'on doit utiliser pour cette exeercice : Cette methode permet de trandformer un mot en vecteur selon son importance dans le dataset -> si le mot est présent réguliérement dans tout les témoignage alors il n'tes que tres peu discriminant alors que a l'inverse si le mot est rare et exclusif a certain témoignages alors il va etre d'autant plus discriminant .

## Probleme de lemmatisation :

Dans ce contexte les vecteur sont precis pour chaque mot différents donc par exemple les mots burn et burning seront compté toatalement différents au yeyx des vecteurs donc il faut utiliser la lemmatisation pour que chaque mots de la meme famille sois normaliser exemple (Burning -> burn) de cette maminere chaque pharse traitatn de brulure peut imorte la conjugaison ect sera traiter avec le meme vecteur pour ce mot précis .

## La methode Spacy vs ntlk

- On aurait pu utiliser la methode fournit par ntlk cependant il faut préciser le type de mot donné pour que la fonction puisse bien focntionner alors que avec Spacy on n'tes pas obliger :

In [26]:
import spacy

nlp = spacy.load("en_core_web_sm")

def lemmatize(text):
    doc = nlp(text)
    return ' '.join([token.lemma_ for token in doc])

dataframe['clean_text'] = dataframe['clean_text'].apply(lemmatize)

print(dataframe[['text', 'clean_text']].head(3))


                                                text  \
0  On uneven paths the heel feels wobbly side‑to‑...   
1  The feverish feeling eased, but I’m now breath...   
2  While working in the garden, I touched a metal...   

                                          clean_text  
0  uneven path heel feel wobbly side‑to‑side . sl...  
1  feverish feeling ease , I ’m breathe fast dry ...  
2  work garden , touch metal grill stove . leave ...  


## Vectorisation

- on va dans cette partie tester les deux methode evoqué durant le kick off : TFIDF et bag of word 

TFIDF designe une methode pour étudié la fréquence d'un mot dans une phrase(tf) et globalement dans un data set(idf)

Bag of Word desinge une methode avec juste une pondération des mots mais pour gerer les mots qui ne sont vbraiment pas assez discriminants on peut utiliser les params de CountVectorizer(max_df=, min_df=) pour supprimer les mots trop occurnets .

## TFIDF

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(dataframe['clean_text'])

print("TF-IDF shape:", tfidf_matrix.shape)
print("Exemple de features:", tfidf_vectorizer.get_feature_names_out()[:10])


TF-IDF shape: (1011, 1733)
Exemple de features: ['abdomen' 'able' 'abruptly' 'accidentally' 'accompany' 'ache' 'achy'
 'across' 'act' 'active']


## Bag Of Word

In [37]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer(max_df=0.7, min_df=2)
bow_matrix = bow_vectorizer.fit_transform(dataframe['clean_text'])

print("BoW shape:", bow_matrix.shape)
print("Exemple de features:", bow_vectorizer.get_feature_names_out()[:10])


BoW shape: (1011, 1109)
Exemple de features: ['abdomen' 'able' 'accidentally' 'accompany' 'ache' 'achy' 'across'
 'active' 'activity' 'actual']


## DBSCAN vs K means

Les deux methode permettre de faire des clustring mais de différentes manières : 

DBscan permet de faire du clustering sans savoir le nombre de groupes a l'avance et ajuste automatiquement tandis que K means permet de faire du clusterinf avec un nombre de cluster donné en params.

Nous allons donc testé ces deux methodes sur nos matrice (BOW et TFIDF)

## Kmeans

In [38]:
from sklearn.cluster import KMeans

# mzthode appliqué a la matrice tfidf
kmeans_tfidf = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans_tfidf.fit(tfidf_matrix)

# methode appliqué a la matrice bow
kmeans_bow = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans_bow.fit(bow_matrix)

print("K-Means TF-IDF - taille des clusters:", np.bincount(kmeans_tfidf.labels_))
print("K-Means BoW    - taille des clusters:", np.bincount(kmeans_bow.labels_))


K-Means TF-IDF - taille des clusters: [112 267 131 295 206]
K-Means BoW    - taille des clusters: [352  86 296 186  91]


## DBSCANs

pour dbscan on a besoin de savoir l'ordre de grandeur des distance entre deux points pour savoir comment manipuler le parmas eps

In [40]:
from sklearn.metrics import pairwise_distances

distances = pairwise_distances(tfidf_matrix, metric='cosine')
print("Distance moyenne:", distances.mean())
print("Distance médiane:", np.median(distances))


Distance moyenne: 0.9714854343737236
Distance médiane: 0.9921771009172138


In [60]:
from sklearn.metrics import pairwise_distances

distances_bow = pairwise_distances(bow_matrix, metric='cosine')
print("Distance moyenne:", distances_bow.mean())
print("Distance médiane:", np.median(distances_bow))


Distance moyenne: 0.9376570509027583
Distance médiane: 0.942833804952497


In [86]:
from sklearn.cluster import DBSCAN

# mzthode appliqué a la matrice tfidf
dbscan_tfidf = DBSCAN(eps=0.76, min_samples=4, metric='cosine')
dbscan_tfidf.fit(tfidf_matrix)

# methode appliqué a la matrice bow
dbscan_bow = DBSCAN(eps=0.66, min_samples=3, metric='cosine')
dbscan_bow.fit(bow_matrix)

print("DBSCAN TF-IDF - nb clusters:", len(set(dbscan_tfidf.labels_) - {-1}), "- bruit:", (dbscan_tfidf.labels_ == -1).sum())
print("DBSCAN BoW    - nb clusters:", len(set(dbscan_bow.labels_) - {-1}), "- bruit:", (dbscan_bow.labels_ == -1).sum())


DBSCAN TF-IDF - nb clusters: 4 - bruit: 186
DBSCAN BoW    - nb clusters: 3 - bruit: 193


## analyse des résultats :

- on constate une vol maximal pour les params choisit cependant trouver ce point d'équiloibre ne signifie pas que les cluster trouvé son bon : il fauit faire une PCA / tsne comme vue dans le kick off pour réduire les dimenssions 

In [92]:
from sklearn.decomposition import PCA

pca = PCA(n_components=500, random_state=42)
tfidf_pca = pca.fit_transform(tfidf_matrix.toarray())

pca_bow = PCA(n_components=500, random_state=42)
bow_pca = pca_bow.fit_transform(bow_matrix.toarray())

print("Variance expliquée:", pca.explained_variance_ratio_.sum())
print("Shape:", tfidf_pca.shape)
print("Shape:", bow_pca.shape)


Variance expliquée: 0.8908020688728899
Shape: (1011, 500)
Shape: (1011, 500)


In [101]:
dbscan_pca = DBSCAN(eps=0.76, min_samples=4, metric='cosine')
dbscan_pca.fit(tfidf_pca)

bow_dbscan_pca = DBSCAN(eps=0.69, min_samples=3, metric='cosine')
bow_dbscan_pca.fit(bow_pca)

print("TFIDF DBSCAN PCA - nb clusters:", len(set(dbscan_pca.labels_) - {-1}), "- bruit:", (dbscan_pca.labels_ == -1).sum())
print("BOW DBSCAN PCA - nb clusters:", len(set(bow_dbscan_pca.labels_) - {-1}), "- bruit:", (bow_dbscan_pca.labels_ == -1).sum())


TFIDF DBSCAN PCA - nb clusters: 3 - bruit: 102
BOW DBSCAN PCA - nb clusters: 3 - bruit: 158
